# DataFlow Sentinel: Job Failure Prediction

<div dir="rtl"><p><strong>עברית:</strong> מחברת זו מתעדת את פיתוח HGB V2 להערכת סיכון לכשל ברמת ריצת ג&#x27;וב. היא כוללת בדיקות נתונים, חלוקה בזמן, השוואת מודלים וניתוח שגיאות. הפלטים השמורים מתעדים את הריצה המקורית; הגרסה הערוכה טרם הורצה מחדש מתחילתה ועד סופה.</p></div>

**English:** This notebook documents HGB V2 development for job-run failure risk assessment, including data checks, temporal splitting, model comparisons, and error analysis. Saved outputs belong to the original run; the edited notebook has not been rerun end-to-end.


In [ ]:
import json
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display

working_dir = Path.cwd().resolve()
project_root = next(
    (p for p in [working_dir, *working_dir.parents]
     if (p / "dbt/dataflow_dbt/dbt_project.yml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from the project or notebooks folder.")

frozen_dir = project_root / "artifacts/models/hgb_v2_20261002T104801_398392Z"
frozen_metadata = json.loads(
    (frozen_dir / "metadata.json").read_text(encoding="utf-8-sig")
)

for package in ("scikit-learn", "numpy", "pandas", "scipy", "joblib"):
    expected = frozen_metadata["versions"][package]
    installed = version(package)
    if installed != expected:
        raise RuntimeError(
            f"{package}: installed {installed}, expected {expected}. "
            "Use the versions in requirements-model.txt."
        )

print("Project:", project_root)
print("Pandas version:", pd.__version__)


## Environment | סביבת העבודה

<div dir="rtl"><p><strong>עברית:</strong> גרסאות ספריות ה-ML נבדקות מול המטאדאטה של המודל השמור. ההרצה דורשת גם PostgreSQL, טבלאות dbt מוכנות ופרטי חיבור מקומיים.</p></div>

**English:** Core ML package versions are checked against the saved model metadata. Execution also requires PostgreSQL, prepared dbt tables, and local connection settings.

[Setup instructions](../README.md) | [Core ML versions](../requirements-model.txt)


In [ ]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
from psycopg.rows import dict_row
import pandas as pd

# project_root was located in the setup cell.
env_path = project_root / ".env"

if not env_path.is_file():
    raise FileNotFoundError(f"Configuration file not found: {env_path}")

config = dotenv_values(env_path)

db_config = {
    "host": "127.0.0.1",
    "port": int(config.get("POSTGRES_PORT") or 5432),
    "dbname": config["POSTGRES_DB"],
    "user": config["POSTGRES_USER"],
    "password": config["POSTGRES_PASSWORD"],
    "connect_timeout": 10,
}

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT * FROM ml.ml_job_features LIMIT 5;")
        job_preview = pd.DataFrame(cur.fetchall())

print("Preview shape:", job_preview.shape)
display(job_preview)

## Data and target | הנתונים והיעד

<div dir="rtl"><p><strong>עברית:</strong> 14,295,731 רשומות משימה אוגדו ל-4,161,396 שורות ג&#x27;וב עם יעד ידוע. היעד failed_job הוא 1 אם לפחות משימה אחת נכשלה, ו-0 אם כל המשימות הסתיימו בסטטוס terminated. מקרים ללא יעד ידוע הוצאו מהאוכלוסייה.</p></div>

**English:** The prepared dataset aggregates 14,295,731 task records into 4,161,396 job rows with known targets. failed_job is 1 when any task failed and 0 when every task has terminated status. Unresolved targets are excluded.

[Job aggregation and target](../dbt/dataflow_dbt/models/gold/fact_job_runs.sql)


In [4]:
quality_query = """
SELECT
    MIN(split_timestamp) AS min_start_time,
    MAX(split_timestamp) AS max_start_time,

    COUNT(*) FILTER (
        WHERE split_timestamp = 0
    ) AS jobs_with_zero_start_time,

    COUNT(*) FILTER (
        WHERE split_timestamp < 0
    ) AS jobs_with_negative_start_time,

    MIN(task_count) AS min_task_count,
    MAX(task_count) AS max_task_count,

    COUNT(*) FILTER (
        WHERE mean_task_plan_cpu IS NULL
           OR max_task_plan_cpu IS NULL
           OR total_task_plan_cpu IS NULL
    ) AS jobs_with_missing_cpu_features,

    COUNT(*) FILTER (
        WHERE mean_task_plan_mem IS NULL
           OR max_task_plan_mem IS NULL
           OR total_task_plan_mem IS NULL
    ) AS jobs_with_missing_memory_features,

    COUNT(*) FILTER (
        WHERE missing_plan_cpu_flag = 1
    ) AS jobs_with_any_missing_task_cpu,

    COUNT(*) FILTER (
        WHERE missing_plan_mem_flag = 1
    ) AS jobs_with_any_missing_task_memory

FROM ml.ml_job_features;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(quality_query)
        quality_result = cur.fetchone()

for check, value in quality_result.items():
    print(f"{check}: {value}")

min_start_time: 0
max_start_time: 691294
jobs_with_zero_start_time: 2286
jobs_with_negative_start_time: 0
min_task_count: 1
max_task_count: 1002
jobs_with_missing_cpu_features: 606
jobs_with_missing_memory_features: 606
jobs_with_any_missing_task_cpu: 4126
jobs_with_any_missing_task_memory: 4126


In [6]:
zero_time_query = """
WITH per_job AS (
    SELECT
        m.job_run_id,
        m.failed_job,
        BOOL_OR(t.start_time > 0) AS has_positive_start,

        COUNT(*) FILTER (
            WHERE t.start_time = 0 AND t.task_status = 'waiting'
        ) AS waiting_at_zero,

        COUNT(*) FILTER (
            WHERE t.start_time = 0 AND t.task_status = 'running'
        ) AS running_at_zero,

        COUNT(*) FILTER (
            WHERE t.start_time = 0 AND t.task_status = 'failed'
        ) AS failed_at_zero,

        COUNT(*) FILTER (
            WHERE t.start_time = 0 AND t.task_status = 'terminated'
        ) AS terminated_at_zero

    FROM ml.ml_job_features AS m
    JOIN staging.stg_batch_tasks AS t
        ON t.job_run_id = m.job_run_id
    WHERE m.split_timestamp = 0
    GROUP BY m.job_run_id, m.failed_job
)
SELECT
    failed_job,
    COUNT(*) AS jobs_at_zero,
    COUNT(*) FILTER (
        WHERE has_positive_start
    ) AS jobs_with_positive_task_starts,
    COUNT(*) FILTER (
        WHERE NOT has_positive_start
    ) AS jobs_without_positive_task_starts,
    SUM(waiting_at_zero) AS waiting_tasks_at_zero,
    SUM(running_at_zero) AS running_tasks_at_zero,
    SUM(failed_at_zero) AS failed_tasks_at_zero,
    SUM(terminated_at_zero) AS terminated_tasks_at_zero
FROM per_job
GROUP BY failed_job
ORDER BY failed_job;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(zero_time_query)
        zero_time_report = pd.DataFrame(cur.fetchall())

print(zero_time_report.set_index("failed_job").T.to_string())

failed_job                           0     1
jobs_at_zero                         9  2277
jobs_with_positive_task_starts       5  2272
jobs_without_positive_task_starts    4     5
waiting_tasks_at_zero                0  3400
running_tasks_at_zero                0     5
failed_tasks_at_zero                 0     5
terminated_tasks_at_zero           485    23


In [7]:
time_fix_query = """
WITH proposed AS (
    SELECT
        m.job_run_id,
        m.failed_job,
        MIN(
            CASE
                WHEN t.task_status = 'waiting'
                     AND t.start_time = 0
                THEN NULL
                ELSE t.start_time
            END
        ) AS proposed_start_time

    FROM ml.ml_job_features AS m
    JOIN staging.stg_batch_tasks AS t
        ON t.job_run_id = m.job_run_id
    WHERE m.split_timestamp = 0
    GROUP BY m.job_run_id, m.failed_job
)
SELECT
    failed_job,
    COUNT(*) AS checked_jobs,
    COUNT(*) FILTER (
        WHERE proposed_start_time > 0
    ) AS jobs_moved_to_positive_time,
    COUNT(*) FILTER (
        WHERE proposed_start_time = 0
    ) AS jobs_still_at_zero,
    COUNT(*) FILTER (
        WHERE proposed_start_time IS NULL
    ) AS jobs_without_start_time
FROM proposed
GROUP BY failed_job
ORDER BY failed_job;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(time_fix_query)
        time_fix_report = pd.DataFrame(cur.fetchall())

print(time_fix_report.set_index("failed_job").T.to_string())

failed_job                   0     1
checked_jobs                 9  2277
jobs_moved_to_positive_time  0  2268
jobs_still_at_zero           9     9
jobs_without_start_time      0     0


In [8]:
remaining_query = """
WITH remaining_tasks AS (
    SELECT
        t.job_run_id,
        g.failed_job,
        t.start_time,
        t.end_time,
        t.task_status,
        (
            t.start_time = 0
            AND t.task_status <> 'waiting'
        ) AS zero_nonwaiting
    FROM gold.fact_job_runs AS g
    JOIN staging.stg_batch_tasks AS t
        ON t.job_run_id = g.job_run_id
    WHERE g.run_start_time = 0
)
SELECT
    job_run_id,
    failed_job,
    MIN(start_time) FILTER (
        WHERE start_time > 0
    ) AS first_positive_start,
    MIN(end_time) FILTER (
        WHERE zero_nonwaiting
    ) AS zero_tasks_min_end,
    MAX(end_time) FILTER (
        WHERE zero_nonwaiting
    ) AS zero_tasks_max_end,
    STRING_AGG(DISTINCT task_status, ', ') FILTER (
        WHERE zero_nonwaiting
    ) AS zero_start_statuses
FROM remaining_tasks
GROUP BY job_run_id, failed_job
ORDER BY failed_job, job_run_id;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(remaining_query)
        remaining_report = pd.DataFrame(cur.fetchall())

print(remaining_report.to_string(index=False))

job_run_id  failed_job  first_positive_start  zero_tasks_min_end  zero_tasks_max_end zero_start_statuses
 j_1097080           0                   NaN                   0                   0          terminated
 j_1212243           0                   NaN                   0                   0          terminated
 j_2476709           0               84340.0                   0              309236          terminated
 j_2778956           0              306470.0                   0              313080          terminated
 j_3238336           0                   NaN                   0                   0          terminated
 j_3478541           0                   NaN                   0                   0          terminated
 j_3499912           0              112872.0                   0              112872          terminated
  j_359051           0              223115.0                   0              311711          terminated
  j_376417           0               88269.0           

## Data quality | איכות הנתונים

<div dir="rtl"><p><strong>עברית:</strong> נבדקים חוסרים בנתוני המשאבים וזמני התחלה לא תקינים. רק ג&#x27;ובים עם split_timestamp גדול מאפס נכללים בחלוקה בזמן.</p></div>

**English:** Checks cover missing resource data and invalid start times. Only jobs with split_timestamp greater than zero enter the temporal split.


In [9]:
split_query = """
SELECT
    COUNT(*) AS eligible_jobs,
    COUNT(*) FILTER (WHERE failed_job = 0) AS successful_jobs,
    COUNT(*) FILTER (WHERE failed_job = 1) AS failed_jobs,
    MIN(split_timestamp) AS min_start_time,
    MAX(split_timestamp) AS max_start_time,

    PERCENTILE_DISC(0.70)
        WITHIN GROUP (ORDER BY split_timestamp) AS train_end_time,

    PERCENTILE_DISC(0.85)
        WITHIN GROUP (ORDER BY split_timestamp) AS validation_end_time

FROM ml.ml_job_features
WHERE split_timestamp > 0;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(split_query)
        split_info = cur.fetchone()

for key, value in split_info.items():
    print(f"{key}: {value}")

eligible_jobs: 4161378
successful_jobs: 4078180
failed_jobs: 83198
min_start_time: 13745
max_start_time: 691294
train_end_time: 520423
validation_end_time: 608501


In [ ]:
split_check_query = """
WITH assigned AS (
    SELECT
        CASE
            WHEN split_timestamp <= %(train_end)s THEN 'Train'
            WHEN split_timestamp <= %(validation_end)s THEN 'Validation'
            ELSE 'Test'
        END AS dataset_split,
        split_timestamp,
        failed_job
    FROM ml.ml_job_features
    WHERE split_timestamp > 0
)
SELECT
    dataset_split,
    COUNT(*) AS jobs,
    MIN(split_timestamp) AS first_start,
    MAX(split_timestamp) AS last_start,
    COUNT(*) FILTER (WHERE failed_job = 1) AS failed_jobs,
    ROUND(100.0 * AVG(failed_job), 3) AS failure_pct
FROM assigned
GROUP BY dataset_split
ORDER BY MIN(split_timestamp);
"""

split_params = {
    "train_end": split_info["train_end_time"],
    "validation_end": split_info["validation_end_time"],
}

if split_params != frozen_metadata["split_params"]:
    raise ValueError("Dataset split boundaries differ from the recorded run.")

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(split_check_query, split_params)
        split_report = pd.DataFrame(cur.fetchall())

print(split_report.to_string(index=False))

In [11]:
label_audit_query = """
WITH task_times AS (
    SELECT
        job_run_id,
        MIN(end_time) FILTER (
            WHERE task_status = 'failed'
              AND start_time > 0
              AND end_time >= start_time
        ) AS failed_at,

        MAX(end_time) FILTER (
            WHERE task_status = 'terminated'
        ) AS succeeded_at,

        COUNT(*) FILTER (
            WHERE task_status = 'terminated'
              AND (
                  start_time IS NULL OR end_time IS NULL
                  OR start_time <= 0 OR end_time < start_time
              )
        ) AS bad_success_times

    FROM staging.stg_batch_tasks
    WHERE task_status IN ('failed', 'terminated')
    GROUP BY job_run_id
),
jobs AS (
    SELECT
        f.split_timestamp,
        f.failed_job,

        CASE WHEN f.split_timestamp <= %(train_end)s
            THEN 'Train' ELSE 'Validation'
        END AS dataset_split,

        CASE WHEN f.split_timestamp <= %(train_end)s
            THEN %(train_end)s ELSE %(validation_end)s
        END AS cutoff,

        CASE
            WHEN f.failed_job = 1 THEN t.failed_at
            WHEN t.bad_success_times = 0 THEN t.succeeded_at
        END AS label_time

    FROM ml.ml_job_features AS f
    LEFT JOIN task_times AS t USING (job_run_id)
    WHERE f.split_timestamp > 0
      AND f.split_timestamp <= %(validation_end)s
)
SELECT
    dataset_split,
    COUNT(*) AS jobs,
    COUNT(*) FILTER (
        WHERE label_time BETWEEN split_timestamp AND cutoff
    ) AS ready_jobs,
    COUNT(*) FILTER (
        WHERE label_time > cutoff
    ) AS late_jobs,
    COUNT(*) FILTER (
        WHERE label_time IS NULL OR label_time < split_timestamp
    ) AS invalid_time_jobs,
    COUNT(*) FILTER (
        WHERE failed_job = 1
          AND label_time BETWEEN split_timestamp AND cutoff
    ) AS ready_failed_jobs
FROM jobs
GROUP BY dataset_split
ORDER BY MIN(split_timestamp);
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(label_audit_query, split_params)
        label_audit = pd.DataFrame(cur.fetchall())

print(label_audit.to_string(index=False))

dataset_split    jobs  ready_jobs  late_jobs  invalid_time_jobs  ready_failed_jobs
        Train 2912966     2912014        920                 32              51731
   Validation  624213      623763        442                  8              26663


## Temporal split | חלוקה בזמן

<div dir="rtl"><p><strong>עברית:</strong> גבולות החלוקה נקבעו באחוזונים 70 ו-85 של זמן ההתחלה: 520423 ו-608501 בריצה המקורית. Train ו-Validation כוללים רק תוויות שזמן זמינותן אינו מאוחר מסוף התקופה המתאימה.</p></div>

**English:** Split boundaries use the 70th and 85th percentiles of start time: 520423 and 608501 in the original run. Train and Validation include only labels available by their respective period cutoffs.


In [12]:
from psycopg.rows import tuple_row

FEATURE_COLUMNS = [
    "task_count",
    "mean_task_plan_cpu",
    "max_task_plan_cpu",
    "total_task_plan_cpu",
    "mean_task_plan_mem",
    "max_task_plan_mem",
    "total_task_plan_mem",
    "missing_plan_cpu_flag",
    "missing_plan_mem_flag",
]

data_columns = [
    "job_run_id", "split_timestamp",
    *FEATURE_COLUMNS, "failed_job"
]

model_dtypes = {column: "float32" for column in FEATURE_COLUMNS}
model_dtypes["failed_job"] = "int8"

selected_features = ", ".join(f"f.{column}" for column in FEATURE_COLUMNS)

load_query = f"""
SELECT
    f.job_run_id,
    f.split_timestamp,
    {selected_features},
    f.failed_job
FROM ml.ml_job_features AS f
JOIN ml.ml_job_label_times AS l USING (job_run_id)
WHERE f.split_timestamp > %(lower_bound)s
  AND f.split_timestamp <= %(upper_bound)s
  AND l.label_available_time
      BETWEEN f.split_timestamp AND %(upper_bound)s
ORDER BY f.split_timestamp, f.job_run_id;
"""


def load_period(name, lower_bound, upper_bound):
    chunks = []
    print(f"Loading {name}...", flush=True)

    params = {
        "lower_bound": int(lower_bound),
        "upper_bound": int(upper_bound),
    }

    with psycopg.connect(**db_config, row_factory=tuple_row) as conn:
        with conn.cursor(name=f"load_{name.lower()}") as cur:
            cur.execute(load_query, params)

            while True:
                rows = cur.fetchmany(50_000)
                if not rows:
                    break

                chunk = pd.DataFrame.from_records(
                    rows, columns=data_columns
                ).astype(model_dtypes)

                chunks.append(chunk)

    if not chunks:
        raise ValueError(f"No rows returned for {name}")

    result = pd.concat(chunks, ignore_index=True)
    print(
        f"{name}: {len(result):,} jobs, "
        f"{int(result['failed_job'].sum()):,} failures"
    )
    return result


train_jobs = load_period(
    "Train", 0, split_params["train_end"]
)

validation_jobs = load_period(
    "Validation",
    split_params["train_end"],
    split_params["validation_end"],
)

# Confirm that loading reproduces the label audit.
assert (len(train_jobs), int(train_jobs["failed_job"].sum())) == (
    2_912_014, 51_731
)
assert (
    len(validation_jobs),
    int(validation_jobs["failed_job"].sum()),
) == (623_763, 26_663)

print("Loaded counts match the label audit.")

Loading Train...
Train: 2,912,014 jobs, 51,731 failures
Loading Validation...
Validation: 623,763 jobs, 26,663 failures
Loaded counts match the label audit.


In [13]:
train_features = train_jobs[FEATURE_COLUMNS]

feature_report = pd.DataFrame({
    "missing": train_features.isna().sum(),
    "infinite": train_features.isin(
        [float("inf"), float("-inf")]
    ).sum(),
    "min": train_features.min(),
    "max": train_features.max(),
})

print(feature_report.to_string())

                       missing  infinite   min            max
task_count                   0         0  1.00    1002.000000
mean_task_plan_cpu         432         0  5.00    1000.000000
max_task_plan_cpu          432         0  5.00    1000.000000
total_task_plan_cpu        432         0  5.00  400400.000000
mean_task_plan_mem         432         0  0.02       9.770000
max_task_plan_mem          432         0  0.02      17.170000
total_task_plan_mem        432         0  0.02    1723.439941
missing_plan_cpu_flag        0         0  0.00       1.000000
missing_plan_mem_flag        0         0  0.00       1.000000


## Model inputs | מאפייני הקלט

<div dir="rtl"><p><strong>עברית:</strong> לאחר הסינון נותרו 2,912,014 ג&#x27;ובים ב-Train ו-623,763 ב-Validation. V1 כולל תשעה מאפיינים של מספר משימות, תכנון CPU וזיכרון ודגלי חוסר. סכומי המשאבים מחברים תוכניות ברמת משימה ואינם מדידות של צריכה בפועל.</p></div>

**English:** Filtering leaves 2,912,014 Train jobs and 623,763 Validation jobs. V1 uses nine task-count, planned-resource, and missingness features. Resource sums aggregate task plans, rather than measured resource consumption.


In [19]:
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

required = ["train_jobs", "validation_jobs", "FEATURE_COLUMNS"]
missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. Run the connection and loading cells."
    )

X_train = train_jobs[FEATURE_COLUMNS]
y_train = train_jobs["failed_job"]

X_val = validation_jobs[FEATURE_COLUMNS]
y_val = validation_jobs["failed_job"]

sentinel_lr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        solver="lbfgs",
        C=1.0,
        class_weight="balanced",
        max_iter=500,
    )),
])

print("scikit-learn:", sklearn.__version__)
print("Training Logistic Regression...", flush=True)

sentinel_lr.fit(X_train, y_train)

print("Training finished.")
print("Iterations:", sentinel_lr.named_steps["model"].n_iter_)

scikit-learn: 1.9.1
Training Logistic Regression...
Training finished.
Iterations: [42]


In [20]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

baseline_scores = np.full(len(y_val), float(y_train.mean()))
lr_scores = sentinel_lr.predict_proba(X_val)[:, 1]


def evaluate_scores(name, y_true, scores, threshold=0.5):
    predictions = (scores >= threshold).astype("int8")

    return {
        "model": name,
        "AP": average_precision_score(y_true, scores),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "alerts": int(predictions.sum()),
    }


validation_results = pd.DataFrame([
    evaluate_scores("Baseline", y_val, baseline_scores),
    evaluate_scores("Logistic Regression", y_val, lr_scores),
])

print(validation_results.round(4).to_string(index=False))

              model     AP  precision  recall     F1  alerts
           Baseline 0.0427     0.0000  0.0000 0.0000       0
Logistic Regression 0.2517     0.1155  0.9347 0.2055  215835


In [21]:
import numpy as np
import pandas as pd

print("Iterations:", sentinel_lr.named_steps["model"].n_iter_)

actual_failure = y_val.to_numpy() == 1
total_failures = int(actual_failure.sum())
threshold_rows = []

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.98, 0.99]:
    predicted_failure = lr_scores >= threshold

    tp = int(np.count_nonzero(predicted_failure & actual_failure))
    fp = int(np.count_nonzero(predicted_failure & ~actual_failure))
    fn = total_failures - tp
    alerts = tp + fp

    precision = tp / alerts if alerts else 0.0
    recall = tp / total_failures if total_failures else 0.0
    denominator = 2 * tp + fp + fn

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "F1": 2 * tp / denominator if denominator else 0.0,
        "alerts": alerts,
        "false_alerts": fp,
        "missed_failures": fn,
    })

threshold_report = pd.DataFrame(threshold_rows)
print(threshold_report.round(4).to_string(index=False))

Iterations: [42]
 threshold  precision  recall     F1  alerts  false_alerts  missed_failures
      0.50     0.1155  0.9347 0.2055  215835        190914             1742
      0.60     0.2413  0.9274 0.3829  102483         77757             1937
      0.70     0.2533  0.9231 0.3975   97170         72557             2050
      0.80     0.1378  0.0350 0.0558    6763          5831            25731
      0.90     0.2059  0.0318 0.0552    4124          3275            25814
      0.95     0.2712  0.0296 0.0533    2906          2118            25875
      0.98     0.3544  0.0275 0.0511    2071          1337            25929
      0.99     0.4123  0.0272 0.0510    1756          1032            25939


In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, confusion_matrix

precision, recall, thresholds = precision_recall_curve(
    y_val,
    lr_scores,
    drop_intermediate=False,
)

# The final precision/recall pair has no matching threshold.
p = precision[:-1]
r = recall[:-1]

f1_values = np.divide(
    2 * p * r,
    p + r,
    out=np.zeros_like(p),
    where=(p + r) > 0,
)

best_idx = int(np.argmax(f1_values))
lr_best_threshold = float(thresholds[best_idx])

predictions = (lr_scores >= lr_best_threshold).astype("int8")

tn, fp, fn, tp = confusion_matrix(
    y_val, predictions, labels=[0, 1]
).ravel()

lr_threshold_result = pd.DataFrame([{
    "precision": p[best_idx],
    "recall": r[best_idx],
    "F1": f1_values[best_idx],
    "alerts": int(tp + fp),
    "true_alerts": int(tp),
    "false_alerts": int(fp),
    "missed_failures": int(fn),
}])

print("Best F1 threshold on Validation:", lr_best_threshold)
print(lr_threshold_result.round(4).to_string(index=False))

Best F1 threshold on Validation: 0.7036485075950623
 precision  recall     F1  alerts  true_alerts  false_alerts  missed_failures
     0.254  0.9229 0.3983   96885        24606         72279             2057


## Baselines | מודלי ייחוס

<div dir="rtl"><p><strong>עברית:</strong> מודלי הייחוס הם ציון קבוע ו-Logistic Regression. השלמת חוסרים וסקיילינג מותאמים על Train בלבד. ההערכה כוללת Average Precision לדירוג, ו-Precision, Recall ו-F1 להחלטות ההתרעה.</p></div>

**English:** References are a constant-score baseline and Logistic Regression. Imputation and scaling are fitted on Train only. Evaluation uses Average Precision for ranking, plus precision, recall, and F1 for alert decisions.


In [23]:
from time import perf_counter
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

sentinel_hgb = HistGradientBoostingClassifier(
    learning_rate=0.1,
    max_iter=100,
    max_leaf_nodes=15,
    min_samples_leaf=200,
    l2_regularization=1.0,
    class_weight="balanced",
    early_stopping=False,
    random_state=42,
)

print("Training HistGradientBoosting...", flush=True)
started = perf_counter()

with threadpool_limits(limits=4):
    sentinel_hgb.fit(X_train, y_train)
    hgb_scores = sentinel_hgb.predict_proba(X_val)[:, 1]

print(f"Training and prediction finished in {perf_counter() - started:.1f} seconds.")
print("Boosting iterations:", sentinel_hgb.n_iter_)

Training HistGradientBoosting...
Training and prediction finished in 30.9 seconds.
Boosting iterations: 100


In [24]:
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, precision_recall_curve


def best_f1_summary(name, y_true, scores):
    precision, recall, thresholds = precision_recall_curve(
        y_true, scores, drop_intermediate=False
    )

    p = precision[:-1]
    r = recall[:-1]

    f1_values = np.divide(
        2 * p * r,
        p + r,
        out=np.zeros_like(p),
        where=(p + r) > 0,
    )

    best_idx = int(np.argmax(f1_values))
    threshold = float(thresholds[best_idx])

    predicted = np.asarray(scores) >= threshold
    actual = np.asarray(y_true) == 1

    alerts = int(predicted.sum())
    tp = int(np.count_nonzero(predicted & actual))

    return {
        "model": name,
        "AP": average_precision_score(y_true, scores),
        "threshold": threshold,
        "precision": p[best_idx],
        "recall": r[best_idx],
        "F1": f1_values[best_idx],
        "alerts": alerts,
        "false_alerts": alerts - tp,
        "missed_failures": int(actual.sum()) - tp,
    }


lr_summary = best_f1_summary(
    "Logistic Regression", y_val, lr_scores
)
hgb_summary = best_f1_summary(
    "HistGradientBoosting", y_val, hgb_scores
)

hgb_best_threshold = hgb_summary["threshold"]

model_comparison = pd.DataFrame([lr_summary, hgb_summary])

print("HGB best F1 threshold:", hgb_best_threshold)
print(model_comparison.round(4).to_string(index=False))

HGB best F1 threshold: 0.8864427085841989
               model     AP  threshold  precision  recall     F1  alerts  false_alerts  missed_failures
 Logistic Regression 0.2517     0.7036     0.2540  0.9229 0.3983   96885         72279             2057
HistGradientBoosting 0.2907     0.8864     0.2808  0.9104 0.4293   86433         62159             2389


In [25]:
import numpy as np
import pandas as pd

actual_failure = np.asarray(y_val) == 1
total_failures = int(actual_failure.sum())
n_jobs = len(actual_failure)

budget_rows = []

for name, model_scores in [
    ("Logistic Regression", lr_scores),
    ("HistGradientBoosting", hgb_scores),
]:
    scores = np.asarray(model_scores, dtype=float)

    # Keep equal scores together when selecting a threshold.
    unique_scores, counts = np.unique(scores, return_counts=True)
    thresholds_desc = unique_scores[::-1]
    cumulative_alerts = np.cumsum(counts[::-1])

    for budget_pct in [1, 5, 10]:
        max_alerts = n_jobs * budget_pct // 100

        candidates = thresholds_desc[
            cumulative_alerts <= max_alerts
        ]
        threshold = (
            float(candidates[-1]) if candidates.size else np.inf
        )

        predicted_failure = scores >= threshold
        alerts = int(predicted_failure.sum())
        tp = int(np.count_nonzero(
            predicted_failure & actual_failure
        ))

        budget_rows.append({
            "model": name,
            "budget_pct": budget_pct,
            "alert_pct": 100 * alerts / n_jobs,
            "threshold": threshold,
            "precision": tp / alerts if alerts else 0.0,
            "recall": tp / total_failures,
            "alerts": alerts,
            "false_alerts": alerts - tp,
            "missed_failures": total_failures - tp,
        })

alert_budget_report = (
    pd.DataFrame(budget_rows)
    .sort_values(["budget_pct", "model"])
)

print(alert_budget_report.round(4).to_string(index=False))

               model  budget_pct  alert_pct  threshold  precision  recall  alerts  false_alerts  missed_failures
HistGradientBoosting           1     0.1287     0.8916     0.8904  0.0268     803            88            25948
 Logistic Regression           1     0.9996     0.8134     0.1472  0.0344    6235          5317            25745
HistGradientBoosting           5     0.1287     0.8916     0.8904  0.0268     803            88            25948
 Logistic Regression           5     1.8162     0.7042     0.0948  0.0403   11329         10255            25589
HistGradientBoosting          10     0.1287     0.8916     0.8904  0.0268     803            88            25948
 Logistic Regression          10     1.8162     0.7042     0.0948  0.0403   11329         10255            25589


In [26]:
import numpy as np
import pandas as pd

# Group by exact scores, before any display rounding.
score_audit = pd.DataFrame({
    "score": np.asarray(hgb_scores),
    "failed_job": np.asarray(y_val),
})

score_groups = (
    score_audit.groupby("score", as_index=False)
    .agg(
        jobs=("failed_job", "size"),
        failures=("failed_job", "sum"),
    )
    .sort_values("score", ascending=False)
    .reset_index(drop=True)
)

score_groups["cumulative_alerts"] = score_groups["jobs"].cumsum()
score_groups["alert_pct"] = (
    100 * score_groups["cumulative_alerts"] / len(y_val)
)

# Find the first score group that exceeds the 10% budget.
budget_limit = len(y_val) * 10 // 100
boundary = int(np.flatnonzero(
    score_groups["cumulative_alerts"].to_numpy() > budget_limit
)[0])

blocking_score = float(score_groups.loc[boundary, "score"])

print("Score groups around the 10% boundary:")
print(
    score_groups.iloc[max(0, boundary - 1):boundary + 2]
    .round(6)
    .to_string(index=False)
)

# Inspect feature combinations inside the blocking group.
group_mask = np.asarray(hgb_scores) == blocking_score

tied_jobs = X_val.loc[group_mask, FEATURE_COLUMNS].copy()
tied_jobs["failed_job"] = np.asarray(y_val)[group_mask]

tied_profiles = (
    tied_jobs.groupby(FEATURE_COLUMNS, dropna=False, as_index=False)
    .agg(
        jobs=("failed_job", "size"),
        failures=("failed_job", "sum"),
    )
    .sort_values("jobs", ascending=False)
)

print("\nBlocking score:", blocking_score)
print("Jobs in this group:", len(tied_jobs))
print("Distinct feature profiles:", len(tied_profiles))
print("\nThree most common feature profiles:")
print(tied_profiles.head(3).to_string(index=False))

Score groups around the 10% boundary:
   score  jobs  failures  cumulative_alerts  alert_pct
0.891605    76         6                803   0.128735
0.888785 85573     23534              86376  13.847567
0.888058    47        22              86423  13.855102

Blocking score: 0.8887850522702704
Jobs in this group: 85573
Distinct feature profiles: 5

Three most common feature profiles:
 task_count  mean_task_plan_cpu  max_task_plan_cpu  total_task_plan_cpu  mean_task_plan_mem  max_task_plan_mem  total_task_plan_mem  missing_plan_cpu_flag  missing_plan_mem_flag  jobs  failures
        1.0               100.0              100.0                100.0                0.10               0.10                 0.10                    0.0                    0.0 85551     23531
        1.0               100.0              100.0                100.0                0.06               0.06                 0.06                    0.0                    0.0     9         2
        1.0               100.0 

In [27]:
from psycopg.rows import dict_row

extra_feature_query = """
SELECT
    t.instance_num,
    t.task_type,
    COUNT(*) AS jobs,
    SUM(f.failed_job) AS failures,
    ROUND(100.0 * AVG(f.failed_job), 2) AS failure_pct

FROM ml.ml_job_features AS f
JOIN ml.ml_job_label_times AS l USING (job_run_id)
JOIN staging.stg_batch_tasks AS t USING (job_run_id)

WHERE f.split_timestamp > %(train_end)s
  AND f.split_timestamp <= %(validation_end)s
  AND l.label_available_time
      BETWEEN f.split_timestamp AND %(validation_end)s

  AND f.task_count = 1
  AND f.mean_task_plan_cpu::real = 100::real
  AND f.mean_task_plan_mem::real = 0.1::real
  AND f.missing_plan_cpu_flag = 0
  AND f.missing_plan_mem_flag = 0

GROUP BY t.instance_num, t.task_type
ORDER BY jobs DESC;
"""

with psycopg.connect(**db_config, row_factory=dict_row) as conn:
    with conn.cursor() as cur:
        cur.execute(extra_feature_query, split_params)
        extra_feature_report = pd.DataFrame(cur.fetchall())

print("Jobs checked:", int(extra_feature_report["jobs"].sum()))
print("Failures:", int(extra_feature_report["failures"].sum()))
print("Different combinations:", len(extra_feature_report))
print(extra_feature_report.head(15).to_string(index=False))

Jobs checked: 85551
Failures: 23531
Different combinations: 470
 instance_num  task_type  jobs  failures failure_pct
            1         10 45191     23349       51.67
            1          1 16966         0        0.00
            2          1  2672         0        0.00
            3         10  1878        57        3.04
            2         10  1674        47        2.81
            3          1  1558         0        0.00
            4         10  1234         8        0.65
            6          1  1051         0        0.00
            5         10   952        63        6.62
            4          1   911         0        0.00
            1         11   715         0        0.00
            5          1   651         1        0.15
            1          8   607         0        0.00
            6         10   374         0        0.00
            9          1   369         0        0.00


## HGB V1 and tied scores | ציונים זהים ב-HGB V1

<div dir="rtl"><p><strong>עברית:</strong> HGB V1 בוחן קשרים לא ליניאריים בין המאפיינים. ב-Validation נמצאה קבוצה גדולה עם אותו ציון: סף יחיד כולל את כולה או מוציא את כולה. הממצא הוביל לבחינת מספר המופעים וסוג המשימה ב-V2.</p></div>

**English:** HGB V1 models nonlinear feature relationships. A large Validation group shares one score, so a threshold includes or excludes the entire group. This finding motivated examining instance counts and task types in V2.


In [ ]:
import numpy as np
import pandas as pd
import psycopg
from psycopg.rows import tuple_row

extra_columns = [
    "job_run_id",
    "total_instances",
    "missing_instance_num_flag",
    "task_type_group",
]

extra_query = """
SELECT
    job_run_id,
    total_instances,
    missing_instance_num_flag,
    task_type_group
FROM ml.ml_job_features_v2
WHERE split_timestamp > 0
  AND split_timestamp <= %(validation_end)s;
"""

chunks = []
print("Loading V2 metadata...", flush=True)

with psycopg.connect(**db_config, row_factory=tuple_row) as conn:
    conn.execute("SET LOCAL max_parallel_workers_per_gather = 0")

    with conn.cursor(name="load_v2_metadata") as cur:
        cur.execute(extra_query, split_params)

        while True:
            rows = cur.fetchmany(50_000)
            if not rows:
                break

            chunk = pd.DataFrame.from_records(
                rows, columns=extra_columns
            ).astype({
                "total_instances": "float64",
                "missing_instance_num_flag": "int8",
            })
            chunks.append(chunk)

if not chunks:
    raise RuntimeError("No V2 metadata was returned.")

v2_metadata = (
    pd.concat(chunks, ignore_index=True)
    .set_index("job_run_id")
)
if not v2_metadata.index.is_unique:
    raise ValueError("Duplicate job_run_id values in V2 metadata.")
del chunks


def make_v2_matrix(jobs, metadata):
    # Match metadata to the exact order of the existing Jobs.
    matched = metadata.reindex(jobs["job_run_id"])

    if matched["task_type_group"].isna().any():
        raise ValueError("Some existing Jobs have no matching V2 metadata.")

    result = pd.concat([
        jobs[FEATURE_COLUMNS].reset_index(drop=True),
        matched.reset_index(drop=True),
    ], axis=1)

    result["task_type_group"] = result["task_type_group"].astype("category")

    if (result["total_instances"] < 0).any():
        raise ValueError("Negative total_instances found; inspect before training.")

    return result


X_train_v2 = make_v2_matrix(train_jobs, v2_metadata)
X_val_v2 = make_v2_matrix(validation_jobs, v2_metadata)

y_train_v2 = train_jobs["failed_job"].to_numpy()
y_val_v2 = validation_jobs["failed_job"].to_numpy()

del v2_metadata

print("Train V2:", X_train_v2.shape)
print("Validation V2:", X_val_v2.shape)

In [29]:
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from threadpoolctl import threadpool_limits

NUMERIC_COLUMNS_V2 = [
    *FEATURE_COLUMNS,
    "total_instances",
    "missing_instance_num_flag",
]

v2_preprocessor = ColumnTransformer([
    ("numeric", "passthrough", NUMERIC_COLUMNS_V2),
    ("task_type", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan,
        dtype=np.float64,
    ), ["task_type_group"]),
])

# Use the same HGB settings as V1.
# The categorical column comes after all numeric columns.
hgb_v2_model = clone(sentinel_hgb).set_params(
    categorical_features=[len(NUMERIC_COLUMNS_V2)]
)

sentinel_hgb_v2 = Pipeline([
    ("preprocessor", v2_preprocessor),
    ("model", hgb_v2_model),
])

print("Training HGB V2...", flush=True)
started = perf_counter()

with threadpool_limits(limits=4):
    sentinel_hgb_v2.fit(X_train_v2, y_train_v2)
    hgb_v2_scores = sentinel_hgb_v2.predict_proba(X_val_v2)[:, 1]

print(f"Training and prediction finished in {perf_counter() - started:.1f} seconds.")

Training HGB V2...
Training and prediction finished in 43.0 seconds.


In [30]:
hgb_v2_summary = best_f1_summary(
    "HGB V2", y_val_v2, hgb_v2_scores
)

hgb_v2_best_threshold = hgb_v2_summary["threshold"]

model_comparison_v2 = pd.DataFrame([
    best_f1_summary("HGB V1", y_val_v2, hgb_scores),
    hgb_v2_summary,
])

print("HGB V2 best F1 threshold:", hgb_v2_best_threshold)
print(model_comparison_v2.round(4).to_string(index=False))

HGB V2 best F1 threshold: 0.9696205830070223
 model     AP  threshold  precision  recall     F1  alerts  false_alerts  missed_failures
HGB V1 0.2907     0.8864     0.2808  0.9104 0.4293   86433         62159             2389
HGB V2 0.5169     0.9696     0.5240  0.9022 0.6629   45910         21854             2607


## HGB V2 | הרחבת מאפייני הקלט

<div dir="rtl"><p><strong>עברית:</strong> V2 מוסיף total_instances, missing_instance_num_flag ו-task_type_group, ובסך הכול 12 מאפיינים. סוג המשימה מטופל כמשתנה קטגוריאלי. פרמטרי HGB נשמרו כפי שהוגדרו ב-V1 כדי לבחון את תרומת המאפיינים הנוספים.</p></div>

**English:** V2 adds total_instances, missing_instance_num_flag, and task_type_group, for 12 features. Task type is treated as categorical. HGB parameters match V1 to assess the contribution of the additional features.


In [32]:
import numpy as np
import pandas as pd
import psycopg
from psycopg.rows import tuple_row

# Freeze the configuration selected using Validation.
FINAL_THRESHOLD = float(hgb_v2_best_threshold)
FINAL_FEATURE_COLUMNS = X_train_v2.columns.tolist()

test_query = """
    select
        f.*,
        l.label_available_time
    from ml.ml_job_features_v2 as f
    left join ml.ml_job_label_times as l
        on f.job_run_id = l.job_run_id
    where f.split_timestamp > %s
    order by f.split_timestamp, f.job_run_id
"""

print("Loading Test V2...")

test_chunks = []

with psycopg.connect(**db_config, row_factory=tuple_row) as conn:
    conn.execute("SET LOCAL max_parallel_workers_per_gather = 0")

    with conn.cursor(name="load_test_v2") as cursor:
        cursor.execute(
            test_query,
            (int(split_params["validation_end"]),),
        )

        column_names = [column.name for column in cursor.description]

        while True:
            rows = cursor.fetchmany(50_000)

            if not rows:
                break

            test_chunks.append(
                pd.DataFrame.from_records(rows, columns=column_names)
            )

if not test_chunks:
    raise ValueError("No Test rows were returned.")

test_jobs_v2 = pd.concat(test_chunks, ignore_index=True)
del test_chunks

if not test_jobs_v2["job_run_id"].is_unique:
    raise ValueError("Duplicate job_run_id values after the Test join.")

missing_label_time = test_jobs_v2["label_available_time"].isna()

invalid_label_time = (
    ~missing_label_time
    & (
        test_jobs_v2["label_available_time"]
        < test_jobs_v2["split_timestamp"]
    )
)

eligible = ~(missing_label_time | invalid_label_time)

print(f"Test jobs before filtering: {len(test_jobs_v2):,}")
print(f"Excluded — missing label time: {missing_label_time.sum():,}")
print(f"Excluded — invalid label time: {invalid_label_time.sum():,}")

test_jobs_v2 = test_jobs_v2.loc[eligible].reset_index(drop=True)

if test_jobs_v2.empty:
    raise ValueError("No eligible Test rows remain.")

X_test_v2 = test_jobs_v2[FINAL_FEATURE_COLUMNS].copy()

# Match training precision, including float32 for the original features.
numeric_dtypes_v2 = {
    column: X_train_v2[column].dtype
    for column in FINAL_FEATURE_COLUMNS
    if column != "task_type_group"
}

X_test_v2 = X_test_v2.astype(numeric_dtypes_v2)
X_test_v2["task_type_group"] = (
    X_test_v2["task_type_group"].astype("category")
)

y_test_v2 = test_jobs_v2["failed_job"].to_numpy(dtype=np.int8)

print("Test V2:", X_test_v2.shape)
print(f"Test failures: {y_test_v2.sum():,}")
print(f"Test failure rate: {y_test_v2.mean():.4%}")
print(f"Frozen threshold: {FINAL_THRESHOLD:.16f}")

Loading Test V2...
Test jobs before filtering: 624,199
Excluded — missing label time: 3
Excluded — invalid label time: 0
Test V2: (624196, 12)
Test failures: 4,790
Test failure rate: 0.7674%
Frozen threshold: 0.9696205830070223


In [33]:
from time import perf_counter
from threadpoolctl import threadpool_limits
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

print("Evaluating the frozen HGB V2 on Test...")
started = perf_counter()

with threadpool_limits(limits=4):
    hgb_v2_test_scores = sentinel_hgb_v2.predict_proba(
        X_test_v2
    )[:, 1]

test_predictions = (
    hgb_v2_test_scores >= FINAL_THRESHOLD
).astype(np.int8)

tn, fp, fn, tp = confusion_matrix(
    y_test_v2,
    test_predictions,
    labels=[0, 1],
).ravel()

test_results = pd.DataFrame([{
    "model": "HGB V2",
    "AP": average_precision_score(y_test_v2, hgb_v2_test_scores),
    "threshold": FINAL_THRESHOLD,
    "precision": precision_score(
        y_test_v2, test_predictions, zero_division=0
    ),
    "recall": recall_score(
        y_test_v2, test_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test_v2, test_predictions, zero_division=0
    ),
    "alerts": int(tp + fp),
    "true_alerts": int(tp),
    "false_alerts": int(fp),
    "missed_failures": int(fn),
    "alert_pct": 100 * float(test_predictions.mean()),
}])

print(f"Finished in {perf_counter() - started:.1f} seconds.")
print(f"Constant-score AP baseline: {y_test_v2.mean():.4f}")
print(
    test_results.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

Evaluating the frozen HGB V2 on Test...
Finished in 3.6 seconds.
Constant-score AP baseline: 0.0077
 model     AP  threshold  precision  recall     F1  alerts  true_alerts  false_alerts  missed_failures  alert_pct
HGB V2 0.1948     0.9696     0.1235  0.5111 0.1989   19826         2448         17378             2342     3.1762


## Threshold and Test | בחירת סף והערכת Test

<div dir="rtl"><p><strong>עברית:</strong> הסף 0.9696205830070223 נבחר לפי F1 ב-Validation והוקפא לפני הערכת Test. התרעה מתקבלת כאשר הציון גדול מהסף או שווה לו. אוכלוסיית Test כוללת 624,196 ג&#x27;ובים לאחר הוצאת שלוש שורות ללא זמן תווית.</p></div>

**English:** The threshold 0.9696205830070223 was selected by Validation F1 and frozen before Test evaluation. An alert is raised when score >= threshold. Test contains 624,196 jobs after excluding three rows without label times.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import json
import platform

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

created_at = datetime.now(timezone.utc)
run_id = created_at.strftime("%Y%m%dT%H%M%S_%fZ")

# Save this execution in a new timestamped directory.
run_dir = project_root / "artifacts" / "models" / f"hgb_v2_{run_id}"
run_dir.mkdir(parents=True, exist_ok=False)

# Preserve Validation and Test results for the same frozen model.
comparison = pd.DataFrame([
    hgb_v2_summary,
    test_results.iloc[0].to_dict(),
])

comparison.insert(0, "split", ["validation", "test"])
comparison["jobs"] = [len(y_val_v2), len(y_test_v2)]
comparison["failure_rate"] = [
    float(np.mean(y_val_v2)),
    float(np.mean(y_test_v2)),
]
comparison["true_alerts"] = (
    comparison["alerts"] - comparison["false_alerts"]
)
comparison["alert_pct"] = (
    100 * comparison["alerts"] / comparison["jobs"]
)

metadata = {
    "model_name": "HGB V2",
    "status": "experimental",
    "created_at_utc": created_at.isoformat(),
    "threshold": float(FINAL_THRESHOLD),
    "threshold_selection": "Maximum F1 on Validation, frozen before Test",
    "prediction_rule": "Alert when score >= threshold",
    "target": "failed_job",
    "positive_class": 1,
    "feature_columns": list(FINAL_FEATURE_COLUMNS),
    "feature_dtypes": {
        column: str(X_train_v2[column].dtype)
        for column in FINAL_FEATURE_COLUMNS
    },
    "split_params": {
        "train_end": int(split_params["train_end"]),
        "validation_end": int(split_params["validation_end"]),
    },
    "rows": {
        "train": int(len(X_train_v2)),
        "validation": int(len(X_val_v2)),
        "test": int(len(X_test_v2)),
    },
    "test_label_time_exclusions": {
        "missing": int(missing_label_time.sum()),
        "invalid": int(invalid_label_time.sum()),
    },
    "evaluation_population": (
        "Jobs with known targets and valid label times; "
        "Train and Validation additionally require label availability "
        "by their respective period cutoffs."
    ),
    "limitations": [
        "Assumes task metadata is available before job start.",
        "Scores have not been calibrated as failure probabilities.",
        "Test performance is substantially lower than Validation.",
    ],
    "versions": {
        "python": platform.python_version(),
        **{
            package: version(package)
            for package in [
                "scikit-learn", "numpy", "pandas", "scipy", "joblib"
            ]
        },
    },
}

bundle = {
    "pipeline": sentinel_hgb_v2,
    "metadata": metadata,
}

model_path = run_dir / "model_bundle.joblib"

joblib.dump(bundle, model_path, compress=3)

(run_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False, allow_nan=False),
    encoding="utf-8",
)

comparison.to_csv(
    run_dir / "validation_test_metrics.csv",
    index=False,
)

# Verify that saving and reloading preserve predictions.
reloaded_bundle = joblib.load(model_path)
check_rows = min(500, len(X_test_v2))

with threadpool_limits(limits=4):
    reloaded_scores = reloaded_bundle["pipeline"].predict_proba(
        X_test_v2.iloc[:check_rows]
    )[:, 1]

np.testing.assert_allclose(
    reloaded_scores,
    hgb_v2_test_scores[:check_rows],
    rtol=0,
    atol=1e-12,
)

print("Model, metadata and metrics saved.")
print("Reload verification passed.")
print("Saved to:", run_dir)

## Model artifact | שמירת המודל

<div dir="rtl"><p><strong>עברית:</strong> נשמרים ה-Pipeline, המאפיינים, הסף, גבולות החלוקה וגרסאות הספריות. טעינה חוזרת נבדקת מול הציונים שחושבו לפני השמירה. כל הרצה יוצרת תיקיית מודל חדשה.</p></div>

**English:** The artifact stores the pipeline, feature schema, threshold, split boundaries, and package versions. Reloaded predictions are compared with the original scores. Each execution creates a new model directory.


In [35]:
import numpy as np
import pandas as pd


def grouped_error_counts(split_name, X, y, scores):
    if not (len(X) == len(y) == len(scores)):
        raise ValueError(f"Length mismatch in {split_name}")

    data = X[
        ["task_type_group", "total_instances", "missing_instance_num_flag"]
    ].reset_index(drop=True).copy()

    data["task_type_group"] = (
        data["task_type_group"].astype("string").fillna("missing")
    )

    data["instance_group"] = pd.cut(
        data["total_instances"],
        bins=[-np.inf, 0, 1, 5, 20, 100, np.inf],
        labels=["<=0", "1", "2-5", "6-20", "21-100", "101+"],
    ).astype("string").fillna("missing")

    # Incomplete instance counts get their own group.
    data.loc[
        data["missing_instance_num_flag"].gt(0),
        "instance_group",
    ] = "missing_or_partial"

    actual = np.asarray(y) == 1
    predicted = np.asarray(scores) >= FINAL_THRESHOLD

    data["failed"] = actual
    data["tp"] = actual & predicted
    data["fp"] = ~actual & predicted
    data["fn"] = actual & ~predicted

    result = (
        data.groupby(
            ["task_type_group", "instance_group"],
            observed=True,
            dropna=False,
        )
        .agg(
            jobs=("failed", "size"),
            failures=("failed", "sum"),
            true_alerts=("tp", "sum"),
            false_alerts=("fp", "sum"),
            missed_failures=("fn", "sum"),
        )
        .reset_index()
    )

    result.insert(0, "split", split_name)
    return result


def add_group_metrics(counts):
    result = counts.copy()
    alerts = result["true_alerts"] + result["false_alerts"]

    result["failure_pct"] = (
        100 * result["failures"] / result["jobs"]
    )
    result["precision_pct"] = (
        100 * result["true_alerts"] / alerts.replace(0, np.nan)
    )
    result["recall_pct"] = (
        100 * result["true_alerts"]
        / result["failures"].replace(0, np.nan)
    )

    return result


profile_counts = pd.concat(
    [
        grouped_error_counts(
            "Validation", X_val_v2, y_val_v2, hgb_v2_scores
        ),
        grouped_error_counts(
            "Test", X_test_v2, y_test_v2, hgb_v2_test_scores
        ),
    ],
    ignore_index=True,
)

count_columns = [
    "jobs", "failures", "true_alerts",
    "false_alerts", "missed_failures",
]

type_counts = (
    profile_counts.groupby(
        ["split", "task_type_group"],
        as_index=False,
        observed=True,
        dropna=False,
    )[count_columns]
    .sum()
)

errors_by_type = add_group_metrics(type_counts).sort_values(
    ["task_type_group", "split"],
    ascending=[True, False],
)

errors_by_profile = add_group_metrics(profile_counts)

# Select groups for descriptive error reporting.
test_groups = errors_by_profile.loc[
    errors_by_profile["split"].eq("Test")
]

profile_keys = ["task_type_group", "instance_group"]

priority_groups = pd.concat(
    [
        test_groups.loc[test_groups["missed_failures"].gt(0)]
        .nlargest(5, "missed_failures"),

        test_groups.loc[test_groups["false_alerts"].gt(0)]
        .nlargest(5, "false_alerts"),
    ],
    ignore_index=True,
)[profile_keys].drop_duplicates()

focused_comparison = (
    errors_by_profile.merge(
        priority_groups,
        on=profile_keys,
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        ["task_type_group", "instance_group", "split"],
        ascending=[True, True, False],
    )
)

metric_columns = [
    "jobs", "failures", "failure_pct",
    "precision_pct", "recall_pct",
    "false_alerts", "missed_failures",
]

print("Performance by task type:")
print(
    errors_by_type[
        ["split", "task_type_group"] + metric_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
        na_rep="—",
    )
)

print("\nGroups with the most Test errors — compared with Validation:")
print(
    focused_comparison[
        ["split"] + profile_keys + metric_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
        na_rep="—",
    )
)

# Save complete summaries alongside the model.
errors_by_type.to_csv(
    run_dir / "error_analysis_by_task_type.csv",
    index=False,
)

errors_by_profile.to_csv(
    run_dir / "error_analysis_by_task_type_and_instances.csv",
    index=False,
)

print("\nError analysis saved to:", run_dir)

Performance by task type:
     split task_type_group   jobs  failures  failure_pct  precision_pct  recall_pct  false_alerts  missed_failures
Validation          type_1 457255      2989         0.65          98.30       23.15            12             2297
      Test          type_1 489204      2532         0.52          99.42       20.34             3             2017
Validation         type_10  76424     23563        30.83          51.67       99.09         21842              214
      Test         type_10  49578      2125         4.29           9.93       90.12         17375              210
Validation         type_11   3484        50         1.44         100.00        6.00             0               47
      Test         type_11   3042        45         1.48         100.00        4.44             0               43
Validation         type_12    740         1         0.14              —        0.00             0                1
      Test         type_12    646         1         0.

In [36]:
compact_errors = focused_comparison[
    [
        "split",
        "task_type_group",
        "instance_group",
        "jobs",
        "failures",
        "failure_pct",
        "precision_pct",
        "recall_pct",
        "false_alerts",
        "missed_failures",
    ]
].rename(columns={
    "task_type_group": "type",
    "instance_group": "instances",
    "failure_pct": "fail_pct",
    "precision_pct": "P_pct",
    "recall_pct": "R_pct",
    "false_alerts": "FP",
    "missed_failures": "FN",
})

print(
    compact_errors.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
        na_rep="—",
    )
)

     split    type instances   jobs  failures  fail_pct  P_pct  R_pct    FP   FN
Validation  type_1         1 115476        46      0.04      —   0.00     0   46
      Test  type_1         1 115051        54      0.05      —   0.00     0   54
Validation  type_1      101+  96698      1899      1.96  93.75   7.90    10 1749
      Test  type_1      101+ 108146      1701      1.57  98.66   8.64     2 1554
Validation  type_1    21-100  57496       254      0.44  94.59  13.78     2  219
      Test  type_1    21-100  63596       224      0.35  97.44  16.96     1  186
Validation  type_1      6-20  74706       258      0.35 100.00  16.67     0  215
      Test  type_1      6-20  85909       208      0.24 100.00  16.35     0  174
Validation type_10         1  55229     23379     42.33  51.67  99.87 21842   30
      Test type_10         1  33429      1946      5.82   9.93  98.41 17375   31
Validation type_10       2-5  11211       179      1.60      —   0.00     0  179
      Test type_10       2-5

## Error analysis | ניתוח שגיאות

<div dir="rtl"><p><strong>עברית:</strong> בריצה המקורית, 17,375 מתוך 17,378 התראות השווא ב-Test היו בג&#x27;ובים מסוג type_10 עם מופע אחד. שיעור הכשל בקבוצה ירד מ-42.33% ב-Validation ל-5.82% ב-Test. הניתוח מתאר שינוי בין התקופות ואינו מוכיח מה גרם לו.</p></div>

**English:** In the recorded run, type_10 jobs with one instance account for 17,375 of 17,378 Test false alerts. Their failure rate falls from 42.33% in Validation to 5.82% in Test. This describes a period shift without establishing its cause.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np

sys.path.insert(0, str(project_root / "src"))

from ml.predictor import JobFailurePredictor

model_dir = run_dir

predictor = JobFailurePredictor(model_dir)

# Compare against scores already computed in the notebook.
positions = np.linspace(
    0,
    len(X_test_v2) - 1,
    num=min(1000, len(X_test_v2)),
    dtype=int,
)

sample = X_test_v2.iloc[positions]
predictions = predictor.predict_batch(sample)
reference_scores = np.asarray(hgb_v2_test_scores)[positions]

assert predictor.threshold == FINAL_THRESHOLD

np.testing.assert_allclose(
    predictions["score"].to_numpy(),
    reference_scores,
    rtol=0,
    atol=1e-12,
)

np.testing.assert_array_equal(
    predictions["alert"].to_numpy(),
    reference_scores >= FINAL_THRESHOLD,
)

print(f"Verified {len(sample):,} predictions against the notebook.")
print("Scores, threshold and alert decisions match.")

# Example of the response for one Job.
single_prediction = predictor.predict_one(
    sample.iloc[0].to_dict()
)

print(json.dumps(single_prediction, indent=2, ensure_ascii=False))

## Inference consistency | עקביות החיזוי

<div dir="rtl"><p><strong>עברית:</strong> נבדקת התאמה בין המחברת למחלקת החיזוי הטוענת את המודל שנשמר באותה הרצה. הבדיקה משווה ציונים, סף והחלטות התרעה בעד 1,000 ג&#x27;ובים.</p></div>

**English:** The check compares notebook predictions with the predictor loading the artifact saved in the same run. Scores, threshold, and alert decisions are compared for up to 1,000 jobs.


## Recorded evaluation results | תוצאות ההערכה השמורות

<div dir="rtl"><p><strong>עברית:</strong> הטבלה מסכמת את מדדי המודל השמור hgb_v2_20261002T104801_398392Z. היא מתעדת את הריצה המקורית ואינה מתעדכנת אוטומטית באימון חדש. AP של מודל בעל ציון קבוע ב-Test הוא 0.0077.</p></div>

**English:** The table summarizes recorded metrics for hgb_v2_20261002T104801_398392Z. It documents the original run and does not update automatically after retraining. The constant-score Test AP baseline is 0.0077.

| Split | Jobs | AP | Precision | Recall | F1 | False alerts | Missed failures |
|---|---:|---:|---:|---:|---:|---:|---:|
| Validation | 623,763 | 0.5169 | 0.5240 | 0.9022 | 0.6629 | 21,854 | 2,607 |
| Test | 624,196 | 0.1948 | 0.1235 | 0.5111 | 0.1989 | 17,378 | 2,342 |


## Application integration | שילוב במערכת

<div dir="rtl"><p><strong>עברית:</strong> המודל השמור משמש את שירות FastAPI ואת ממשק Streamlit. בדיקות מומלצות מבוססות על כללים, ללא תרומות מאפיינים לתחזית. הסימולטור העסקי מציג תרחישים המבוססים על הנחות עלות ומניעה.</p></div>

**English:** The saved model serves the FastAPI service and Streamlit interface. Recommended checks are rule-based; feature attributions are not available. The business simulator presents scenarios based on assumed costs and prevention rates.

[API documentation](../docs/api_v2.md) | [Dashboard documentation](../docs/dashboard_v2.md)


## Knowledge assistant | עוזר הידע

<div dir="rtl"><p><strong>עברית:</strong> עוזר הידע מאחזר קטעים מכרטיס המודל ומנסח תשובות עם מקורות באמצעות Ollama. המאגר הנוכחי כולל מסמך אחד ושישה קטעים. אין בו לוגים של ג&#x27;ובים; תפקידו להסביר את התיעוד.</p></div>

**English:** The assistant retrieves model-card excerpts and generates cited answers through Ollama. Its current corpus is one document with six chunks. It explains documentation and contains no individual job logs.

[Model card](../docs/knowledge_base/hgb_v2_model_card.md) | [RAG implementation](../scripts/ask_rag.py)


## Limitations and next steps | מגבלות והמשך עבודה

<div dir="rtl"><p><strong>עברית:</strong> המודל ניסיוני: זמינות המאפיינים לפני ריצה והפרדת ריצות באמצעות מזהה הג&#x27;וב עדיין דורשות אימות. הציון אינו הסתברות מכוילת, ותועלת תפעולית לא אומתה. הסף נבחר לפי F1; התאמה לעלויות ולקיבולת צוות היא עבודה להמשך. שינויים בעקבות ניתוח ה-Test דורשים הערכה על תקופה נוספת שלא שימשה לפיתוח.</p></div>

**English:** The model is experimental: pre-run feature availability and job-key semantics still require verification. Scores are uncalibrated and operational utility is unverified. The F1-selected threshold still needs cost-and-capacity assessment. Test-informed changes require another evaluation period that has not been used for development.

[Implementation status and remaining requirements](../docs/implementation_alignment_v3.md)
